# 05. Target-Review Signal Extraction and Direct-Cue Audit — Herbal Supplements

This notebook defines the target-review-only evidence layer used before synthetic-query generation. It reads the eligible case reserve from Notebook 03 and preserves the user, target item, target timestamp, regime, and target-selection identity of every case.

The held-out target review is the only positive evidence source. Target-item metadata is joined only as a negative cue dictionary used to remove exact title, Brand, identifier, seller or manufacturer, and package or dosage cues. Metadata cannot add, repair, or supplement query content.

The extraction identifies category-specific signals for ingredient or herb, need or benefit, form, claim or dietary constraint, and flavor, and maps them to the shared semantic-role schema. Flavor is retained as an audited signal family; downstream query construction determines whether it enters the active query seed.

Prior user history, population review-derived item signals, ratings, helpful votes, sentiment, language-model outputs, and target metadata are not permitted as positive query evidence. Raw target-review text is not exported. The intended outputs are bounded signal columns, coverage summaries, insufficiency diagnostics, direct-cue audits, and provenance manifests.

The received notebook contains no stored execution. Its implementation and validation status are stated separately below.


## Execution and Authority Status

All 16 code cells in the received file have null execution counts and no stored outputs. This file therefore documents an intended implementation but does not by itself evidence the reported thesis run. It must not replace any frozen benchmark artifact unless it is deliberately executed and all identity, timestamp, evidence-boundary, direct-cue, and output checks pass.

One executable manifest string describes the Notebook 03 source as having “dynamic balanced source counts.” This is legacy wording: Notebook 03 exports an unbalanced eligible reserve of 24,590 cold, 3,703 weak, and 670 strong cases, while final balancing occurs downstream. Because that wording is contained in an executable string literal, it remains unchanged in this documentation-only task and must be reported as an unresolved provenance-description mismatch.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# ==== Define Inputs, Outputs, and Evidence Policies ====
import json
import os
import re
from pathlib import Path

import numpy as np
import pandas as pd

CATEGORY_ID = "herbal"
CATEGORY_FOLDER = "herbal_supplements"
CATEGORY_LABEL = "Herbal Supplements"

PROJECT_ROOT = Path("/content/drive/MyDrive/thesis_recsys/categories") / CATEGORY_FOLDER
os.chdir(PROJECT_ROOT)

SAMPLED_QUERY_CASES_PATH = PROJECT_ROOT / "data/interim/user_regime_sampling/herbal_sampled_query_cases.parquet"
ITEM_SCHEMA_PATH = PROJECT_ROOT / "data/processed/items/herbal_item_schema.parquet"

OUTPUT_DIR = PROJECT_ROOT / "data/interim/review_signal_extraction"
REVIEW_SIGNAL_DIR = PROJECT_ROOT / "data/processed/review_signals"

SIGNAL_PARQUET_PATH = OUTPUT_DIR / "herbal_review_signal_extraction.parquet"
SIGNAL_CSV_PATH = OUTPUT_DIR / "herbal_review_signal_extraction.csv"
SIGNAL_PARQUET_COMPAT_PATH = REVIEW_SIGNAL_DIR / "herbal_review_signals.parquet"
SIGNAL_CSV_COMPAT_PATH = REVIEW_SIGNAL_DIR / "herbal_review_signals.csv"
REGIME_SUMMARY_CSV_PATH = OUTPUT_DIR / "herbal_review_signal_regime_summary.csv"
FAMILY_COVERAGE_SUMMARY_CSV_PATH = OUTPUT_DIR / "herbal_review_signal_family_coverage_summary.csv"
INSUFFICIENT_REASON_SUMMARY_CSV_PATH = OUTPUT_DIR / "herbal_review_signal_insufficient_reason_summary.csv"
STRONG_INSUFFICIENT_AUDIT_CSV_PATH = OUTPUT_DIR / "herbal_review_signal_strong_insufficient_audit.csv"
DIRECT_CUE_AUDIT_CSV_PATH = OUTPUT_DIR / "herbal_review_signal_direct_cue_audit.csv"
MANIFEST_JSON_PATH = OUTPUT_DIR / "herbal_review_signal_extraction_manifest.json"
SUMMARY_JSON_PATH = OUTPUT_DIR / "herbal_review_signal_extraction_summary.json"
SAMPLING_MANIFEST_PATH = PROJECT_ROOT / "data/interim/user_regime_sampling/herbal_user_regime_sampling_manifest.json"

REGIME_ORDER = ["cold", "weak", "strong"]
MAX_TARGET_RANK_ALLOWED = 5

EXPECTED_TOTAL_N = None
EXPECTED_REGIME_COUNTS = None
EXPECTED_TARGET_SELECTION_MODE = "recent_eligible_review_rank_le5"

SOURCE_ELIGIBLE_POOL_ROLE = "notebook03_source_case_pool"

QUERY_EVIDENCE_SOURCE = "target_review_only"
EXPECTED_MIN_QUERY_SAFE_TOKEN_COUNT = 2
EXPECTED_MIN_QUERY_SAFE_SIGNAL_FAMILIES = 1
EXPECTED_MIN_QUERY_SAFE_SIGNAL_TOTAL = 1

HARMONIZED_REQUIRED_ITEM_SCHEMA_COLS = [
    "parent_asin",
    "title",
    "facet_brand_text",
    "identifier_diagnostic_text",
    "facet_policy_version",
    "brand_policy",
    "identifier_policy",
]

HARMONIZED_FACET_POLICY_VERSION = "harmonized_v2_global_review_brand_retrieval_profile"
EXPECTED_BRAND_POLICY = "separate_preference_facet__query_unsafe__retrieval_profile_graph_safe"
EXPECTED_IDENTIFIER_POLICY = "diagnostic_only__excluded_from_query_safe_profile_and_canonical_retrieval_text"

PACKAGE_QUANTITY_POLICY = "scrub_numeric_unit_patterns_keep_rows"
PACKAGE_QUANTITY_NUMERIC_UNIT_PATTERN_VERSION = "numeric_unit_scrub_v1"
PACKAGE_QUANTITY_ROWS_REMOVED = 0

PACKAGE_QUANTITY_UNIT_TERMS = [
    "capsule", "capsules", "cap", "caps",
    "tablet", "tablets", "pill", "pills",
    "softgel", "softgels", "gummy", "gummies",
    "serving", "servings", "dose", "doses", "dosage",
    "drop", "drops", "dropper", "droppers",
    "packet", "packets", "sachet", "sachets",
    "bottle", "bottles", "jar", "jars",
    "bag", "bags", "box", "boxes",
    "oz", "ounce", "ounces", "fl oz",
    "mg", "milligram", "milligrams",
    "mcg", "microgram", "micrograms",
    "g", "gram", "grams", "kg",
    "ml", "milliliter", "milliliters",
    "liter", "liters",
    "calorie", "calories", "carb", "carbs",
    "sodium", "spf",
]

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
REVIEW_SIGNAL_DIR.mkdir(parents=True, exist_ok=True)

print("Input path:", SAMPLED_QUERY_CASES_PATH)
print("Output path:", SIGNAL_PARQUET_COMPAT_PATH)

In [ ]:
# ==== Run Synthetic Identity-Contract Tests ====
# These low-cost tests verify one row per case, complete reserve coverage,
# and unchanged user, target, timestamp, regime, and selection-mode identities.

def sc0_assert_notebook05_source_identity_contract(source_df, output_df):
    required_source_cols = {
        "case_id", "user_id", "target_parent_asin", "target_timestamp_ms",
        "regime", "target_selection_mode",
    }
    required_output_cols = set(required_source_cols)
    missing_source = sorted(required_source_cols - set(source_df.columns))
    missing_output = sorted(required_output_cols - set(output_df.columns))
    if missing_source or missing_output:
        raise AssertionError(
            f"Missing SC-0 identity columns: source={missing_source}, output={missing_output}"
        )

    source = source_df[list(required_source_cols)].copy()
    output = output_df[list(required_output_cols)].copy()
    for frame_name, frame in [("source", source), ("output", output)]:
        frame["case_id"] = frame["case_id"].astype(str).str.strip()
        if frame["case_id"].eq("").any():
            raise AssertionError(f"{frame_name} contains empty case_id values.")
        if frame["case_id"].duplicated().any():
            duplicated = sorted(frame.loc[frame["case_id"].duplicated(), "case_id"].unique().tolist())
            raise AssertionError(f"{frame_name} duplicates case_id values: {duplicated}")

    source_ids = set(source["case_id"])
    output_ids = set(output["case_id"])
    lost_ids = sorted(source_ids - output_ids)
    extra_ids = sorted(output_ids - source_ids)
    if lost_ids or extra_ids:
        raise AssertionError(
            f"Notebook 05 output must exactly cover the Notebook 03 eligible reserve; "
            f"lost={lost_ids}, extra={extra_ids}"
        )

    identity_cols = ["user_id", "target_parent_asin", "target_timestamp_ms", "regime", "target_selection_mode"]
    source_key = source.set_index("case_id")[identity_cols].astype(str).sort_index()
    output_key = output.set_index("case_id")[identity_cols].astype(str).sort_index()
    mismatch = source_key.ne(output_key)
    if mismatch.any().any():
        bad_cases = sorted(mismatch.index[mismatch.any(axis=1)].tolist())
        raise AssertionError(f"Notebook 05 target/regime/selection identity changed for cases: {bad_cases}")


def _sc0_expect_failure(label, source_df, output_df):
    try:
        sc0_assert_notebook05_source_identity_contract(source_df, output_df)
    except AssertionError:
        return
    raise AssertionError(f"SC-0 negative test did not fail: {label}")


_sc0_source = pd.DataFrame([
    {
        "case_id": "herbal_case_001", "user_id": "u1", "target_parent_asin": "A1",
        "target_timestamp_ms": 101, "regime": "cold",
        "target_selection_mode": EXPECTED_TARGET_SELECTION_MODE,
    },
    {
        "case_id": "herbal_case_002", "user_id": "u2", "target_parent_asin": "A2",
        "target_timestamp_ms": 202, "regime": "weak",
        "target_selection_mode": EXPECTED_TARGET_SELECTION_MODE,
    },
])
_sc0_output = _sc0_source.copy()
sc0_assert_notebook05_source_identity_contract(_sc0_source, _sc0_output)
_sc0_expect_failure("duplicate case_id", pd.concat([_sc0_source, _sc0_source.iloc[[0]]], ignore_index=True), _sc0_output)
_sc0_expect_failure("required reserve case lost", _sc0_source, _sc0_output.iloc[[0]].copy())
_sc0_changed_target = _sc0_output.copy()
_sc0_changed_target.loc[_sc0_changed_target["case_id"].eq("herbal_case_002"), "target_parent_asin"] = "A2_CHANGED"
_sc0_expect_failure("target identity changed", _sc0_source, _sc0_changed_target)
print("SC-0 Notebook 05 synthetic input-schema identity tests: PASS")


In [ ]:
# ==== Declare Shared Review-Signal Roles and Query Boundaries ====
COMMON_REVIEW_SIGNAL_FRAMEWORK_VERSION = "common_review_signal_framework_v2_query_audit"

COMMON_REVIEW_SIGNAL_FAMILY_TO_ROLE = {'ingredient_or_herb': 'ingredient_or_composition', 'benefit_need': 'need_benefit_concern', 'form': 'form_texture', 'claim_diet': 'claim_constraint', 'flavor': 'sensory'}

HERBAL_REVIEW_CATEGORY_ANCHORS = {"herbal", "supplement", "herbal supplement", "dietary supplement"}
HERBAL_REVIEW_GENERIC_UTILITY_TOKENS = {
    "support", "supports", "help", "helps", "promote", "promotes", "boost",
    "formula", "blend", "complex", "product", "solution",
}
HERBAL_REVIEW_CONTEXT_DEPENDENT_TOKENS = {"daily", "natural", "wellness", "health", "care", "routine"}
HERBAL_SPECIFIC_SUPPORT_NEEDS = {
    "immune", "immunity", "digestive", "digestion", "sleep", "joint",
    "stress", "energy", "liver", "stomach", "gut", "focus", "memory",
    "inflammation", "urinary", "throat", "detox", "bladder", "calm",
}
HERBAL_FORM_TOKENS = {
    "capsule", "capsules", "gummy", "gummies", "powder", "tablet", "tablets",
    "drop", "drops", "softgel", "tincture", "syrup", "liquid", "tea",
}
HERBAL_MULTIWORD_ENTITY_FRAGMENT_TOKENS = {
    "milk", "olive", "saw", "seed", "dong", "john", "biloba", "holy",
    "evening", "horny", "vera", "red", "black", "fruit",
}  # Warning-only; compare with the full source signal span before changing normalization.

COMMON_REVIEW_SIGNAL_CONTRACT = {
    "category_id": CATEGORY_ID if "CATEGORY_ID" in globals() else "herbal",
    "category_folder": CATEGORY_FOLDER,
    "category_label": CATEGORY_LABEL,
    "framework_version": COMMON_REVIEW_SIGNAL_FRAMEWORK_VERSION,
    "family_to_common_role": COMMON_REVIEW_SIGNAL_FAMILY_TO_ROLE,
    "category_anchors": sorted(HERBAL_REVIEW_CATEGORY_ANCHORS),
    "generic_utility_tokens": sorted(HERBAL_REVIEW_GENERIC_UTILITY_TOKENS),
    "context_dependent_tokens": sorted(HERBAL_REVIEW_CONTEXT_DEPENDENT_TOKENS),
    "specific_support_needs": sorted(HERBAL_SPECIFIC_SUPPORT_NEEDS),
    "possible_multiword_entity_fragment_tokens": sorted(HERBAL_MULTIWORD_ENTITY_FRAGMENT_TOKENS),
    "brand_signal_policy": "Brand is not extracted as a functional review-derived signal. Exact brand text is used only for direct-cue leakage removal unless a validated brand-signal experiment explicitly enables it.",
    "brand_used_for_signal_extraction": False,
    "query_evidence_source": QUERY_EVIDENCE_SOURCE,
    "item_metadata_evidence_used": False,
    "historical_review_evidence_used": False,
    "user_prior_evidence_used": False,
    "query_safe_residual_column": "query_safe_text",
    "policy_note": "Review-signal dictionaries and thresholds remain category-specific. Item metadata is used only to remove direct title, brand, and identifier shortcuts from target-review-derived query-safe text.",
}


In [ ]:
# ==== Define Text Cleaning and Signal Dictionaries ====
def normalize_space(value):
    if value is None:
        return ''

    if isinstance(value, (list, tuple, set, np.ndarray, pd.Series)):
        return ' | '.join(
            text for text in (normalize_space(v) for v in value)
            if text
        )

    if isinstance(value, dict):
        return ' | '.join(
            text for text in (normalize_space(v) for v in value.values())
            if text
        )

    try:
        if pd.isna(value):
            return ''
    except Exception:
        pass

    text = re.sub(r'\s+', ' ', str(value).replace('\n', ' ').replace('\t', ' ')).strip()

    if text.lower() in {'', 'none', 'null', 'nan', 'n/a', 'na', '[]', '{}'}:
        return ''

    return text


def tokenize_text(text):
    return re.findall(r"[A-Za-z0-9']+", normalize_space(text).lower())


def unique_keep_order(values):
    ordered_values = []
    seen_values = set()

    for value in values:
        normalized_value = normalize_space(value).lower()

        if normalized_value and normalized_value not in seen_values:
            seen_values.add(normalized_value)
            ordered_values.append(normalized_value)

    return ordered_values


def split_pipe_values(value):
    normalized_value = normalize_space(value)

    if not normalized_value:
        return []

    return unique_keep_order(re.split(r'\s*\|\s*', normalized_value))


def join_signals(values):
    return ' | '.join(unique_keep_order(values))


ASIN_PATTERN = re.compile(
    r'\bb0[a-z0-9]{8}\b',
    re.IGNORECASE,
)

PACKAGE_QUANTITY_PATTERN = re.compile(
    (
        r'\b'
        r'\d+(?:\.\d+)?\s?'
        r'(?:mg|mcg|iu|g|gram|grams|ml|oz|fl\.?\s?oz|ct|count|capsule|capsules|'
        r'tablet|tablets|softgel|softgels|gummy|gummies|serving|servings|pack|packs|'
        r'bottle|bottles)'
        r'\b'
    ),
    re.IGNORECASE,
)

DIRECT_IDENTIFIER_PATTERN = re.compile(
    r'\b(asin|sku|upc|barcode|seller|manufacturer)\b',
    re.IGNORECASE,
)

SELLER_MANUFACTURER_PATTERN = re.compile(
    r'\b(sold by|seller|manufacturer|made by|distributed by|shipped by)\b',
    re.IGNORECASE,
)

AMAZON_URL_FRAGMENT_SEQUENCE_PATTERN = re.compile(
    (
        r'\b(?:https?|www|amazon|com|gp|product|dp|ref|ppx|yo|dt|b|search|asin|title|ie|utf8|th|'
        r'ref_|qid|sr|keywords|psc|sprefix|crid|pd|rd|i)\b'
        r'(?:\s+\b(?:https?|www|amazon|com|gp|product|dp|ref|ppx|yo|dt|b|search|asin|title|ie|utf8|th|'
        r'ref_|qid|sr|keywords|psc|sprefix|crid|pd|rd|i|\d+)\b)+'
    ),
    re.IGNORECASE,
)

AMAZON_URL_FRAGMENT_TOKEN_PATTERN = re.compile(
    r'\b(?:https?|www|amazon|com|gp|product|dp|ref|ppx|yo|dt|b|search|asin|title|ie|utf8|th)\b',
    re.IGNORECASE,
)

DIRECT_CUE_METADATA_COLS = [
    'itemctx_facet_brand_text',
    'itemctx_identifier_diagnostic_text',
    'target_parent_asin',
]

TITLE_AUDIT_METADATA_COLS = [
    'itemctx_title',
]


def cue_tokens_from_metadata(row):
    cue_text_parts = []

    for col in DIRECT_CUE_METADATA_COLS:
        value = row.get(col, '')
        normalized_value = normalize_space(value)

        if normalized_value:
            cue_text_parts.extend(split_pipe_values(normalized_value))

    cue_tokens = set()

    for text in cue_text_parts:
        for token in tokenize_text(text):
            if len(token) >= 3:
                cue_tokens.add(token)

    return cue_tokens

def remove_exact_phrases(text, phrases):
    cleaned_text = normalize_space(text)

    for phrase in phrases:
        phrase_tokens = tokenize_text(phrase)
        if not phrase_tokens:
            continue

        phrase_pattern = r'[^a-z0-9]+'.join(
            re.escape(token)
            for token in phrase_tokens
        )
        cleaned_text = re.sub(
            r'(?<![a-z0-9])' + phrase_pattern + r'(?![a-z0-9])',
            ' ',
            cleaned_text,
            flags=re.IGNORECASE,
        )

    return normalize_space(cleaned_text)


def remove_direct_item_cues(text, row):
    cleaned_text = normalize_space(text)

    cleaned_text = ASIN_PATTERN.sub(' ', cleaned_text)
    cleaned_text = AMAZON_URL_FRAGMENT_SEQUENCE_PATTERN.sub(' ', cleaned_text)
    cleaned_text = AMAZON_URL_FRAGMENT_TOKEN_PATTERN.sub(' ', cleaned_text)
    cleaned_text = DIRECT_IDENTIFIER_PATTERN.sub(' ', cleaned_text)
    cleaned_text = SELLER_MANUFACTURER_PATTERN.sub(' ', cleaned_text)

    title_phrases = [
        row.get(col, '')
        for col in TITLE_AUDIT_METADATA_COLS
        if normalize_space(row.get(col, ''))
    ]
    cleaned_text = remove_exact_phrases(cleaned_text, title_phrases)

    cue_tokens = cue_tokens_from_metadata(row)

    safe_tokens = [
        token for token in tokenize_text(cleaned_text)
        if token not in cue_tokens
    ]

    return ' '.join(safe_tokens)

def exact_phrase_present(text, phrase):
    normalized_phrase = normalize_space(phrase).lower()

    if len(normalized_phrase) < 4:
        return False

    normalized_text = normalize_space(text).lower()

    return bool(
        re.search(
            r'(?<![a-z0-9])' + re.escape(normalized_phrase) + r'(?![a-z0-9])',
            normalized_text,
        )
    )


def any_exact_phrase_present(text, row, cols):
    for col in cols:
        if col in row.index and exact_phrase_present(text, row.get(col, '')):
            return True

    return False


SIGNAL_PATTERNS = {
    'ingredient_or_herb': {
        'turmeric': [r'\bturmeric\b'],
        'curcumin': [r'\bcurcumin\b'],
        'ginger': [r'\bginger\b'],
        'elderberry': [r'\belderberry\b'],
        'ashwagandha': [r'\bashwagandha\b'],
        'mushroom': [r'\bmushrooms?\b'],
        "lion's mane": [r"\blion'?s mane\b"],
        'reishi': [r'\breishi\b'],
        'chaga': [r'\bchaga\b'],
        'milk thistle': [r'\bmilk thistle\b'],
        'echinacea': [r'\bechinacea\b'],
        'ginseng': [r'\bginseng\b'],
        'cranberry': [r'\bcranberry\b'],
        'peppermint': [r'\bpeppermint\b'],
        'chamomile': [r'\bchamomile\b'],
        'valerian': [r'\bvalerian\b'],
        'garlic': [r'\bgarlic\b'],
        'berberine': [r'\bberberine\b'],
        'maca': [r'\bmaca\b'],
        'moringa': [r'\bmoringa\b'],
        'saw palmetto': [r'\bsaw palmetto\b'],
        'black seed': [r'\bblack seed\b'],
    },
    'benefit_need': {
        'immune': [r'\bimmune\b'],
        'immunity': [r'\bimmunity\b'],
        'sleep': [r'\bsleep\b'],
        'stress': [r'\bstress\b'],
        'calm': [r'\bcalm\b'],
        'relax': [r'\brelax\w*\b'],
        'digestion': [r'\bdigestion\b', r'\bdigest\w*\b'],
        'stomach': [r'\bstomach\b'],
        'gut': [r'\bgut\b'],
        'energy': [r'\benergy\b'],
        'focus': [r'\bfocus\b'],
        'memory': [r'\bmemory\b'],
        'joint': [r'\bjoint\b'],
        'inflammation': [r'\binflammation\b'],
        'liver': [r'\bliver\b'],
        'detox': [r'\bdetox\b'],
        'throat': [r'\bthroat\b'],
        'urinary': [r'\burinary\b'],
        'bladder': [r'\bbladder\b'],
    },
    'form': {
        'capsule': [r'\bcapsules?\b'],
        'tablet': [r'\btablets?\b'],
        'softgel': [r'\bsoftgels?\b'],
        'gummy': [r'\bgumm(?:y|ies)\b'],
        'powder': [r'\bpowder\b'],
        'tea': [r'\btea\b'],
        'liquid extract': [r'\bliquid extract\b'],
        'tincture': [r'\btincture\b'],
        'drop': [r'\bdrops?\b'],
    },
    'claim_diet': {
        'organic': [r'\borganic\b'],
        'vegan': [r'\bvegan\b'],
        'vegetarian': [r'\bvegetarian\b'],
        'non-gmo': [r'\bnon[- ]?gmo\b'],
        'gluten-free': [r'\bgluten[- ]?free\b'],
        'sugar-free': [r'\bsugar[- ]?free\b'],
        'caffeine-free': [r'\bcaffeine[- ]?free\b'],
        'dairy-free': [r'\bdairy[- ]?free\b'],
        'soy-free': [r'\bsoy[- ]?free\b'],
    },
    'flavor': {
        'berry': [r'\bberry\b'],
        'lemon': [r'\blemon\b'],
        'orange': [r'\borange\b'],
        'mint': [r'\bmint\b'],
        'honey': [r'\bhoney\b'],
    },
}

SIGNAL_FAMILY_CONFIG = [
    {
        'family': 'ingredient_or_herb',
        'signal_col': 'ingredient_or_herb_signals',
        'flag_col': 'has_ingredient_or_herb',
    },
    {
        'family': 'benefit_need',
        'signal_col': 'benefit_need_signals',
        'flag_col': 'has_benefit_need',
    },
    {
        'family': 'form',
        'signal_col': 'form_signals',
        'flag_col': 'has_form',
    },
    {
        'family': 'claim_diet',
        'signal_col': 'claim_diet_signals',
        'flag_col': 'has_claim_diet',
    },
    {
        'family': 'flavor',
        'signal_col': 'flavor_signals',
        'flag_col': 'has_flavor',
    },
]

SIGNAL_FAMILIES = [
    config['family']
    for config in SIGNAL_FAMILY_CONFIG
]

SIGNAL_COLUMNS = [
    config['signal_col']
    for config in SIGNAL_FAMILY_CONFIG
]

SIGNAL_FLAG_COLUMNS = [
    config['flag_col']
    for config in SIGNAL_FAMILY_CONFIG
]

# =========================================================
# Remove package and dosage quantities from query-safe text
# =========================================================

PACKAGE_QUANTITY_NUMERIC_UNIT_RE = re.compile(
    r"""
    \b
    (?:
        \d+(?:[.,]\d+)?              # 1000, 25, 0.5, 10.25
        |
        half|quarter|one|two|three|four|five|six|seven|eight|nine|ten
    )
    \s*
    (?:
        mg|mcg|g|kg|ml|l|oz|fl\s*oz|iu
        |milligram(?:s)?
        |microgram(?:s)?
        |gram(?:s)?
        |kilogram(?:s)?
        |milliliter(?:s)?
        |liter(?:s)?
        |ounce(?:s)?
        |capsule(?:s)?
        |cap(?:s)?
        |tablet(?:s)?
        |pill(?:s)?
        |softgel(?:s)?
        |gumm(?:y|ies)
        |serving(?:s)?
        |dose(?:s)?
        |dosage
        |drop(?:s)?
        |dropper(?:s)?
        |packet(?:s)?
        |sachet(?:s)?
        |bottle(?:s)?
        |jar(?:s)?
        |bag(?:s)?
        |box(?:es)?
        |calorie(?:s)?
        |carb(?:s)?
        |sodium
    )
    \b
    """,
    flags=re.IGNORECASE | re.VERBOSE,
)

PACKAGE_QUANTITY_SPF_RE = re.compile(
    r"""
    \b
    spf
    \s*
    \d+(?:[.,]\d+)?
    \b
    """,
    flags=re.IGNORECASE | re.VERBOSE,
)

PACKAGE_QUANTITY_STANDALONE_MEASURE_RE = re.compile(
    r"""
    \b
    (?:
        \d+(?:[.,]\d+)?\s*(?:x|×)\s*\d+(?:[.,]\d+)?
        |
        \d+\s*/\s*\d+
    )
    \b
    """,
    flags=re.IGNORECASE | re.VERBOSE,
)

WHITESPACE_RE = re.compile(r"\s+")


def scrub_numeric_unit_patterns_keep_rows(text):
    text = normalize_space(text)
    before = text

    text = PACKAGE_QUANTITY_NUMERIC_UNIT_RE.sub(" ", text)
    text = PACKAGE_QUANTITY_SPF_RE.sub("spf", text)
    text = PACKAGE_QUANTITY_STANDALONE_MEASURE_RE.sub(" ", text)
    text = PACKAGE_QUANTITY_PATTERN.sub(" ", text)
    text = WHITESPACE_RE.sub(" ", text).strip()

    return text, int(text != before)


def apply_package_quantity_scrub(df, text_col):
    if text_col not in df.columns:
        raise RuntimeError(f"Missing required text column for package quantity scrub: {text_col}")

    out = df.copy()

    before_col = f"{text_col}_before_package_quantity_scrub"
    flag_col = f"{text_col}_package_quantity_scrub_applied"
    token_delta_col = f"{text_col}_package_quantity_scrub_token_delta"

    out[before_col] = out[text_col].fillna("").astype(str).map(normalize_space)

    scrubbed = out[before_col].map(scrub_numeric_unit_patterns_keep_rows)
    out[text_col] = scrubbed.map(lambda x: x[0])
    out[flag_col] = scrubbed.map(lambda x: x[1]).astype(int)

    before_token_n = out[before_col].map(lambda x: len(tokenize_text(x)))
    after_token_n = out[text_col].fillna("").astype(str).map(lambda x: len(tokenize_text(x)))
    out[token_delta_col] = (before_token_n - after_token_n).astype(int)

    return out


def build_package_quantity_scrub_summary(df, text_col):
    flag_col = f"{text_col}_package_quantity_scrub_applied"
    token_delta_col = f"{text_col}_package_quantity_scrub_token_delta"

    if flag_col not in df.columns:
        raise RuntimeError(f"Missing package scrub flag column: {flag_col}")

    if token_delta_col not in df.columns:
        raise RuntimeError(f"Missing package scrub token delta column: {token_delta_col}")

    return {
        "package_quantity_policy": PACKAGE_QUANTITY_POLICY,
        "package_quantity_numeric_unit_pattern_version": PACKAGE_QUANTITY_NUMERIC_UNIT_PATTERN_VERSION,
        "package_quantity_rows_removed": int(PACKAGE_QUANTITY_ROWS_REMOVED),
        "package_quantity_text_repair_applied": True,
        "package_quantity_scrubbed_rows": int(df[flag_col].fillna(0).astype(int).sum()),
        "package_quantity_scrubbed_rate": float(df[flag_col].fillna(0).astype(float).mean() if len(df) else 0.0),
        "package_quantity_scrub_token_delta_total": int(df[token_delta_col].fillna(0).astype(int).sum()),
    }


In [ ]:
# ==== Load Eligible Cases and Target-Item Cue Dictionaries ====
required_signal_input_cols = [
    'case_id',
    'user_id',
    'target_parent_asin',
    'target_timestamp_ms',
    'target_review_datetime',
    'target_rank_desc',
    'target_selection_mode',
    'regime',
    'prior_history_n',
    'query_safe_token_count',
    'query_safe_signal_family_count',
    'query_safe_signal_total_count',
    'query_safe_text',
    'query_convertible_flag',
]

item_context_source_cols = [
    'parent_asin',
    'title',
    'facet_brand_text',
    'identifier_diagnostic_text',
    'facet_policy_version',
    'brand_policy',
    'identifier_policy',
]

sampled_query_cases_df = pd.read_parquet(SAMPLED_QUERY_CASES_PATH)
item_schema_df = pd.read_parquet(ITEM_SCHEMA_PATH)
if SAMPLING_MANIFEST_PATH.exists():
    with open(SAMPLING_MANIFEST_PATH, "r", encoding="utf-8") as f:
        sampling_manifest = json.load(f)

missing_signal_input_cols = [
    col for col in required_signal_input_cols
    if col not in sampled_query_cases_df.columns
]

missing_item_context_source_cols = [
    col for col in item_context_source_cols
    if col not in item_schema_df.columns
]

if missing_signal_input_cols:
    raise RuntimeError(f'Missing sampled query case columns: {missing_signal_input_cols}')

if missing_item_context_source_cols:
    raise RuntimeError(f'Missing item context source columns: {missing_item_context_source_cols}')

missing_harmonized_cols = [
    col for col in HARMONIZED_REQUIRED_ITEM_SCHEMA_COLS
    if col not in item_schema_df.columns
]

if missing_harmonized_cols:
    raise RuntimeError(
        'Notebook 05 expects the harmonized Notebook 03 item schema. '
        f'Missing required columns: {missing_harmonized_cols}'
    )

if item_schema_df['parent_asin'].isna().any():
    raise RuntimeError('Item schema contains null parent_asin values.')

item_schema_df['parent_asin'] = item_schema_df['parent_asin'].astype(str).map(normalize_space)

if item_schema_df['parent_asin'].eq('').any():
    raise RuntimeError('Item schema contains empty parent_asin values.')

if item_schema_df['parent_asin'].duplicated().any():
    duplicated_item_count = int(item_schema_df['parent_asin'].duplicated().sum())
    raise RuntimeError(f'Item schema contains duplicated parent_asin rows: {duplicated_item_count}')

brand_policy_values = set(
    item_schema_df['brand_policy'].dropna().astype(str).unique().tolist()
)
identifier_policy_values = set(
    item_schema_df['identifier_policy'].dropna().astype(str).unique().tolist()
)
facet_policy_values = set(
    item_schema_df['facet_policy_version'].dropna().astype(str).unique().tolist()
)

if brand_policy_values != {EXPECTED_BRAND_POLICY}:
    raise RuntimeError(f'Unexpected brand_policy values in item schema: {brand_policy_values}')

if identifier_policy_values != {EXPECTED_IDENTIFIER_POLICY}:
    raise RuntimeError(f'Unexpected identifier_policy values in item schema: {identifier_policy_values}')

if facet_policy_values != {HARMONIZED_FACET_POLICY_VERSION}:
    raise RuntimeError(f'Unexpected facet_policy_version values in item schema: {facet_policy_values}')

actual_total_n = int(len(sampled_query_cases_df))
sampled_regime_counts = (
    sampled_query_cases_df['regime']
    .value_counts()
    .reindex(REGIME_ORDER, fill_value=0)
    .astype(int)
    .to_dict()
)

EXPECTED_TOTAL_N = int(actual_total_n)
EXPECTED_REGIME_COUNTS = dict(sampled_regime_counts)

manifest_source_counts = sampling_manifest.get(
    "eligible_pool_regime_counts_before_query_balance",
    sampling_manifest.get("eligible_pool_regime_counts"),
)
if manifest_source_counts is not None:
    manifest_source_counts = {
        regime: int(manifest_source_counts.get(regime, 0))
        for regime in REGIME_ORDER
    }
    if manifest_source_counts != EXPECTED_REGIME_COUNTS:
        raise RuntimeError(
            "Notebook 03 source-pool regime counts differ from the sampling manifest: "
            f"actual={EXPECTED_REGIME_COUNTS}, manifest={manifest_source_counts}"
        )

manifest_source_total = sampling_manifest.get(
    "eligible_pool_total_n_before_query_balance",
    sampling_manifest.get("eligible_pool_rows"),
)
if manifest_source_total is not None and int(manifest_source_total) != EXPECTED_TOTAL_N:
    raise RuntimeError(
        "Notebook 03 source-pool row count differs from the sampling manifest: "
        f"actual={EXPECTED_TOTAL_N}, manifest={int(manifest_source_total)}"
    )

required_non_null_source_cols = [
    'case_id',
    'user_id',
    'target_parent_asin',
    'target_timestamp_ms',
    'regime',
]
for col in required_non_null_source_cols:
    if sampled_query_cases_df[col].isna().any():
        missing_count = int(sampled_query_cases_df[col].isna().sum())
        raise RuntimeError(f'Source case column {col} must be non-null. Missing rows: {missing_count}.')

invalid_regimes = sorted(set(sampled_query_cases_df['regime'].dropna().astype(str)) - set(REGIME_ORDER))
if invalid_regimes:
    raise RuntimeError(f'Source case regime must be cold, weak, or strong. Invalid values: {invalid_regimes}.')

if sampled_query_cases_df['case_id'].astype(str).map(normalize_space).eq('').any():
    raise RuntimeError('Source case_id must be non-empty.')

if sampled_query_cases_df['user_id'].astype(str).map(normalize_space).eq('').any():
    raise RuntimeError('Source user_id must be non-empty.')

if sampled_query_cases_df['target_parent_asin'].astype(str).map(normalize_space).eq('').any():
    raise RuntimeError('Source target_parent_asin must be non-empty.')

if not sampled_query_cases_df.loc[
    sampled_query_cases_df["regime"].eq("cold"),
    "prior_history_n",
].eq(0).all():
    raise RuntimeError("Cold source rows must have prior_history_n == 0.")

if not sampled_query_cases_df.loc[
    sampled_query_cases_df["regime"].eq("weak"),
    "prior_history_n",
].between(1, 4).all():
    raise RuntimeError("Weak source rows must have 1 <= prior_history_n <= 4.")

if not sampled_query_cases_df.loc[
    sampled_query_cases_df["regime"].eq("strong"),
    "prior_history_n",
].ge(5).all():
    raise RuntimeError("Strong source rows must have prior_history_n >= 5.")

if not sampled_query_cases_df['target_rank_desc'].le(MAX_TARGET_RANK_ALLOWED).all():
    raise RuntimeError(f'All target_rank_desc values must be <= {MAX_TARGET_RANK_ALLOWED}.')

if not sampled_query_cases_df['target_selection_mode'].astype(str).eq(EXPECTED_TARGET_SELECTION_MODE).all():
    observed_modes = sorted(sampled_query_cases_df['target_selection_mode'].dropna().astype(str).unique().tolist())
    raise RuntimeError(
        f'target_selection_mode must be {EXPECTED_TARGET_SELECTION_MODE}; observed {observed_modes}.'
    )

missing_target_timestamp_count = int(sampled_query_cases_df['target_timestamp_ms'].isna().sum())
if missing_target_timestamp_count > 0:
    raise RuntimeError(
        f'target_timestamp_ms must be present and non-null. Missing rows: {missing_target_timestamp_count}.'
    )

if not sampled_query_cases_df['query_convertible_flag'].eq(1).all():
    raise RuntimeError('All sampled query cases must be query-convertible.')
if sampled_query_cases_df['query_safe_token_count'].lt(EXPECTED_MIN_QUERY_SAFE_TOKEN_COUNT).any():
    raise RuntimeError('Notebook 03 query-safe token threshold mismatch.')
if sampled_query_cases_df['query_safe_signal_family_count'].lt(EXPECTED_MIN_QUERY_SAFE_SIGNAL_FAMILIES).any():
    raise RuntimeError('Notebook 03 signal-family threshold mismatch.')
if sampled_query_cases_df['query_safe_signal_total_count'].lt(EXPECTED_MIN_QUERY_SAFE_SIGNAL_TOTAL).any():
    raise RuntimeError('Notebook 03 signal-total threshold mismatch.')

if sampled_query_cases_df['query_safe_text'].map(normalize_space).eq('').any():
    raise RuntimeError('query_safe_text must be non-empty for every sampled case.')

if sampled_query_cases_df['case_id'].duplicated().any():
    duplicated_case_count = int(sampled_query_cases_df['case_id'].duplicated().sum())
    raise RuntimeError(f'Sampled query cases duplicate case_id rows: {duplicated_case_count}')

if sampled_query_cases_df['user_id'].duplicated().any():
    duplicated_user_count = int(sampled_query_cases_df['user_id'].duplicated().sum())
    raise RuntimeError(f'Sampled query cases duplicate user_id rows: {duplicated_user_count}')

target_key_cols = ['user_id', 'target_parent_asin', 'target_timestamp_ms']

if sampled_query_cases_df[target_key_cols].duplicated().any():
    duplicated_target_count = int(sampled_query_cases_df[target_key_cols].duplicated().sum())
    raise RuntimeError(
        f'Sampled query cases duplicate selected targets per user: {duplicated_target_count}'
    )

sampled_query_cases_df = sampled_query_cases_df.copy()
sampled_query_cases_df['target_parent_asin'] = (
    sampled_query_cases_df['target_parent_asin'].astype(str).map(normalize_space)
)

item_context_lookup_df = item_schema_df[item_context_source_cols].copy()
item_context_lookup_df['target_parent_asin'] = (
    item_context_lookup_df['parent_asin'].astype(str).map(normalize_space)
)

item_context_lookup_df = (
    item_context_lookup_df
    .drop(columns=['parent_asin'])
    .drop_duplicates('target_parent_asin', keep='first')
    .rename(
        columns={
            col: f'itemctx_{col}'
            for col in item_context_source_cols
            if col != 'parent_asin'
        }
    )
)

sampled_query_cases_df = sampled_query_cases_df.merge(
    item_context_lookup_df,
    on='target_parent_asin',
    how='left',
    indicator='item_metadata_merge_status',
    validate='many_to_one',
)

sampled_query_cases_df['item_metadata_exists'] = (
    sampled_query_cases_df['item_metadata_merge_status'].eq('both')
)

metadata_matched_rows = int(sampled_query_cases_df['item_metadata_exists'].sum())
metadata_match_rate = metadata_matched_rows / len(sampled_query_cases_df)

if metadata_matched_rows != EXPECTED_TOTAL_N:
    unmatched_metadata_preview = sampled_query_cases_df.loc[
        ~sampled_query_cases_df['item_metadata_exists'],
        ['case_id', 'user_id', 'target_parent_asin', 'item_metadata_merge_status'],
    ].head(20)

    raise RuntimeError(
        f'Item metadata merge matched {metadata_matched_rows}/{EXPECTED_TOTAL_N} rows. '
        f'Preview: {unmatched_metadata_preview.to_dict(orient="records")}'
    )

query_safe_text_before_repair = sampled_query_cases_df['query_safe_text'].map(normalize_space)

sampled_query_cases_df['query_safe_text'] = sampled_query_cases_df.apply(
    lambda row: remove_direct_item_cues(query_safe_text_before_repair.loc[row.name], row),
    axis=1,
)

query_safe_text_after_first_repair = sampled_query_cases_df['query_safe_text'].map(normalize_space)

sampled_query_cases_df['query_safe_text'] = sampled_query_cases_df.apply(
    lambda row: remove_direct_item_cues(query_safe_text_after_first_repair.loc[row.name], row),
    axis=1,
)

query_safe_text_after_second_repair = sampled_query_cases_df['query_safe_text'].map(normalize_space)

query_safe_text_repaired_mask = query_safe_text_before_repair.ne(query_safe_text_after_second_repair)
query_safe_text_second_pass_repair_mask = query_safe_text_after_first_repair.ne(query_safe_text_after_second_repair)

sampled_query_cases_df['direct_cue_repair_applied'] = query_safe_text_repaired_mask.astype(bool)
sampled_query_cases_df['direct_cue_repair_count'] = (
    query_safe_text_before_repair.ne(query_safe_text_after_first_repair).astype(int)
    + query_safe_text_second_pass_repair_mask.astype(int)
)

sampled_query_cases_df = apply_package_quantity_scrub(
    sampled_query_cases_df,
    text_col='query_safe_text',
)

# Repeat the package and dosage scrub and verify that no removable cue remains.
query_safe_text_after_package_scrub = sampled_query_cases_df["query_safe_text"].map(normalize_space)
query_safe_text_after_final_package_scrub = query_safe_text_after_package_scrub.map(
    lambda text: scrub_numeric_unit_patterns_keep_rows(text)[0]
)
final_package_scrub_mask = query_safe_text_after_package_scrub.ne(
    query_safe_text_after_final_package_scrub
)

if final_package_scrub_mask.any():
    before_token_n = query_safe_text_after_package_scrub.map(lambda text: len(tokenize_text(text)))
    after_token_n = query_safe_text_after_final_package_scrub.map(lambda text: len(tokenize_text(text)))

    sampled_query_cases_df["query_safe_text"] = query_safe_text_after_final_package_scrub
    sampled_query_cases_df["query_safe_text_package_quantity_scrub_applied"] = (
        sampled_query_cases_df["query_safe_text_package_quantity_scrub_applied"].fillna(0).astype(int)
        | final_package_scrub_mask.astype(int)
    ).astype(int)
    sampled_query_cases_df["query_safe_text_package_quantity_scrub_token_delta"] = (
        sampled_query_cases_df["query_safe_text_package_quantity_scrub_token_delta"].fillna(0).astype(int)
        + (before_token_n - after_token_n).astype(int)
    )

remaining_package_quantity_mask = sampled_query_cases_df["query_safe_text"].map(
    lambda text: bool(PACKAGE_QUANTITY_PATTERN.search(normalize_space(text)))
)
if remaining_package_quantity_mask.any():
    remaining_preview = sampled_query_cases_df.loc[
        remaining_package_quantity_mask,
        ["case_id", "user_id", "target_parent_asin", "query_safe_text"],
    ].head(20)
    raise RuntimeError(
        "Package quantity cue remains after final scrub. "
        f"Preview:\n{remaining_preview}"
    )

package_quantity_scrub_summary = build_package_quantity_scrub_summary(
    sampled_query_cases_df,
    text_col='query_safe_text',
)

package_quantity_scrub_applied = (
    sampled_query_cases_df['query_safe_text_package_quantity_scrub_applied'].astype(int)
    if 'query_safe_text_package_quantity_scrub_applied' in sampled_query_cases_df.columns
    else pd.Series(0, index=sampled_query_cases_df.index)
)
sampled_query_cases_df['direct_cue_repair_count'] = (
    sampled_query_cases_df['direct_cue_repair_count'].astype(int)
    + package_quantity_scrub_applied
)
sampled_query_cases_df['direct_cue_repair_applied'] = sampled_query_cases_df['direct_cue_repair_count'].gt(0)

review_text_source_col = 'target_review_text' if 'target_review_text' in sampled_query_cases_df.columns else 'query_safe_text'
sampled_query_cases_df['target_review_token_count'] = sampled_query_cases_df[review_text_source_col].map(
    lambda value: len(tokenize_text(normalize_space(value)))
).astype(int)

if sampled_query_cases_df['query_safe_text'].eq('').any():
    emptied_query_preview = sampled_query_cases_df.loc[
        sampled_query_cases_df['query_safe_text'].eq(''),
        ['case_id', 'user_id', 'target_parent_asin'],
    ].head(10)

    raise RuntimeError(
        'Brand/package/identifier leakage repair emptied query_safe_text rows: '
        f'{emptied_query_preview.to_dict(orient="records")}'
    )

review_reputation_cols_present = [
    col for col in item_schema_df.columns
    if (
        str(col).startswith('historical_review_')
        or str(col).startswith('review_reputation_')
        or str(col) == 'review_reputation_facet_text'
    )
]

itemctx_title_non_empty = int(
    sampled_query_cases_df['itemctx_title']
    .map(normalize_space)
    .ne('')
    .sum()
)

itemctx_facet_brand_text_non_empty = int(
    sampled_query_cases_df['itemctx_facet_brand_text']
    .map(normalize_space)
    .ne('')
    .sum()
)

itemctx_identifier_diagnostic_text_non_empty = int(
    sampled_query_cases_df['itemctx_identifier_diagnostic_text']
    .map(normalize_space)
    .ne('')
    .sum()
)

print('Input row count:', len(sampled_query_cases_df))


In [ ]:
# ==== Extract Review-Safe Signal Families ====
def extract_family_signals(text, family_name):
    normalized_text = normalize_space(text).lower()
    extracted_signals = []

    for label, patterns in SIGNAL_PATTERNS[family_name].items():
        if any(re.search(pattern, normalized_text) for pattern in patterns):
            extracted_signals.append(label)

    return extracted_signals


def build_extraction_insufficient_reason(
    query_safe_token_count,
    signal_family_count,
    signal_total_count,
    possible_entity_fragment_flag,
    complete_multiword_entity_count,
):
    reasons = []
    if query_safe_token_count < EXPECTED_MIN_QUERY_SAFE_TOKEN_COUNT:
        reasons.append('low_query_safe_token_count')
    if signal_family_count < EXPECTED_MIN_QUERY_SAFE_SIGNAL_FAMILIES:
        reasons.append('low_signal_family_count')
    if signal_total_count < EXPECTED_MIN_QUERY_SAFE_SIGNAL_TOTAL:
        reasons.append('low_signal_total_count')
    if possible_entity_fragment_flag and complete_multiword_entity_count == 0:
        reasons.append('possible_entity_fragment_without_complete_entity')
    return ' | '.join(reasons) if reasons else 'sufficient'


if 'sampled_query_cases_df' not in globals():
    raise RuntimeError('sampled_query_cases_df is not defined. Run the input loading cell first.')

required_signal_source_cols = [
    'case_id',
    'user_id',
    'target_parent_asin',
    'target_timestamp_ms',
    'target_review_datetime',
    'target_rank_desc',
    'target_selection_mode',
    'regime',
    'prior_history_n',
    'query_safe_text',
]

missing_signal_source_cols = [
    col for col in required_signal_source_cols
    if col not in sampled_query_cases_df.columns
]

if missing_signal_source_cols:
    raise RuntimeError(f'Missing signal source columns: {missing_signal_source_cols}')

signal_rows = []

for row in sampled_query_cases_df.itertuples(index=False):
    row_dict = row._asdict()
    query_safe_text = normalize_space(row_dict['query_safe_text'])

    family_signal_map = {
        config['family']: extract_family_signals(query_safe_text, config['family'])
        for config in SIGNAL_FAMILY_CONFIG
    }

    signal_family_count = int(
        sum(len(family_signal_map[config['family']]) > 0 for config in SIGNAL_FAMILY_CONFIG)
    )
    signal_total_count = int(
        sum(len(family_signal_map[config['family']]) for config in SIGNAL_FAMILY_CONFIG)
    )
    query_safe_token_count = int(len(tokenize_text(query_safe_text)))
    ingredient_signals = family_signal_map.get('ingredient_or_herb', [])
    possible_entity_fragment_flag = int(
        any(signal.lower() in HERBAL_MULTIWORD_ENTITY_FRAGMENT_TOKENS for signal in ingredient_signals)
    )
    complete_multiword_entity_count = int(
        sum((' ' in signal or "'" in signal) for signal in ingredient_signals)
    )
    review_signal_extraction_sufficient = bool(
        query_safe_token_count >= EXPECTED_MIN_QUERY_SAFE_TOKEN_COUNT
        and signal_family_count >= EXPECTED_MIN_QUERY_SAFE_SIGNAL_FAMILIES
        and signal_total_count >= EXPECTED_MIN_QUERY_SAFE_SIGNAL_TOTAL
    )
    extraction_insufficient_reason = build_extraction_insufficient_reason(
        query_safe_token_count=query_safe_token_count,
        signal_family_count=signal_family_count,
        signal_total_count=signal_total_count,
        possible_entity_fragment_flag=bool(possible_entity_fragment_flag),
        complete_multiword_entity_count=complete_multiword_entity_count,
    )

    signal_row = {
        'case_id': row_dict['case_id'],
        'user_id': row_dict['user_id'],
        'target_parent_asin': row_dict['target_parent_asin'],
        'target_timestamp_ms': int(row_dict['target_timestamp_ms']),
        'target_review_datetime': row_dict['target_review_datetime'],
        'target_rank_desc': int(row_dict['target_rank_desc']),
        'target_selection_mode': row_dict['target_selection_mode'],
        'regime': row_dict['regime'],
        'query_safe_text': query_safe_text,
        'query_safe_residual_text': query_safe_text,
        'signal_source': 'heldout_target_review',
        'query_evidence_source': QUERY_EVIDENCE_SOURCE,
        'item_metadata_evidence_used': False,
        'historical_review_evidence_used': False,
        'user_prior_evidence_used': False,
        'target_metadata_fallback_used': False,
        'item_context_fallback_used': False,
        'rating_evidence_used': False,
        'sentiment_evidence_used': False,
        'target_review_token_count': int(row_dict.get('target_review_token_count', query_safe_token_count)),
        'query_safe_token_count': query_safe_token_count,
        'query_safe_text_package_quantity_scrub_applied': int(row_dict.get('query_safe_text_package_quantity_scrub_applied', 0)),
        'query_safe_text_package_quantity_scrub_token_delta': int(row_dict.get('query_safe_text_package_quantity_scrub_token_delta', 0)),
        'direct_cue_repair_applied': bool(row_dict.get('direct_cue_repair_applied', False)),
        'direct_cue_repair_count': int(row_dict.get('direct_cue_repair_count', 0)),
        'direct_cue_warning_count': 0,
        'signal_family_count': signal_family_count,
        'signal_total_count': signal_total_count,
        'query_safe_signal_family_count': signal_family_count,
        'query_safe_signal_total_count': signal_total_count,
        'review_signal_extraction_sufficient': review_signal_extraction_sufficient,
        'review_only_query_evidence_sufficient': review_signal_extraction_sufficient,
        'extraction_insufficient_reason': extraction_insufficient_reason,
        'possible_entity_fragment_flag': possible_entity_fragment_flag,
    }

    for config in SIGNAL_FAMILY_CONFIG:
        family = config['family']
        signal_col = config['signal_col']
        flag_col = config['flag_col']
        family_signals = family_signal_map[family]

        signal_row[signal_col] = join_signals(family_signals)
        signal_row[flag_col] = int(len(family_signals) > 0)
        signal_row[f'{family}_signal_count'] = int(len(family_signals))

    signal_rows.append(signal_row)

review_signals_df = pd.DataFrame(signal_rows)

expected_signal_output_cols = (
    [
        'case_id',
        'user_id',
        'target_parent_asin',
        'target_timestamp_ms',
        'target_review_datetime',
        'target_rank_desc',
        'target_selection_mode',
        'regime',
        'query_safe_text',
        'query_safe_residual_text',
        'signal_source',
        'query_evidence_source',
        'item_metadata_evidence_used',
        'historical_review_evidence_used',
        'user_prior_evidence_used',
        'target_metadata_fallback_used',
        'item_context_fallback_used',
        'rating_evidence_used',
        'sentiment_evidence_used',
        'target_review_token_count',
        'query_safe_token_count',
        'signal_family_count',
        'signal_total_count',
        'query_safe_signal_family_count',
        'query_safe_signal_total_count',
        'review_signal_extraction_sufficient',
        'review_only_query_evidence_sufficient',
        'extraction_insufficient_reason',
        'possible_entity_fragment_flag',
        'direct_cue_repair_applied',
        'direct_cue_repair_count',
        'direct_cue_warning_count',
    ]
    + SIGNAL_COLUMNS
    + SIGNAL_FLAG_COLUMNS
)

missing_signal_output_cols = [
    col for col in expected_signal_output_cols
    if col not in review_signals_df.columns
]

if missing_signal_output_cols:
    raise RuntimeError(f'Missing signal output columns: {missing_signal_output_cols}')

if len(review_signals_df) != len(sampled_query_cases_df):
    raise RuntimeError(
        f'Signal row count mismatch: {len(review_signals_df)} vs {len(sampled_query_cases_df)}'
    )

print('Output row count:', len(review_signals_df))
print('Sufficient count:', int(review_signals_df['review_signal_extraction_sufficient'].sum()))
print('Insufficient count:', int((~review_signals_df['review_signal_extraction_sufficient']).sum()))


In [ ]:
# ==== Map Herbal Signals to Shared Semantic Roles ====
_review_signal_frame_name = "review_signals_df"
_review_signal_frame = globals().get(_review_signal_frame_name)
if _review_signal_frame is None:
    raise RuntimeError(f"{_review_signal_frame_name} must exist before common review signal annotation.")

if "common_framework_version" not in _review_signal_frame.columns:
    _review_signal_frame["common_framework_version"] = COMMON_REVIEW_SIGNAL_FRAMEWORK_VERSION
if "review_signal_source_scope" not in _review_signal_frame.columns:
    _review_signal_frame["review_signal_source_scope"] = "target_review_query_safe_text"
if "is_downstream_query_safe" not in _review_signal_frame.columns:
    _review_signal_frame["is_downstream_query_safe"] = 1
if "common_signal_family_count" not in _review_signal_frame.columns:
    source_col = "query_safe_signal_family_count" if "query_safe_signal_family_count" in _review_signal_frame.columns else "signal_family_count"
    _review_signal_frame["common_signal_family_count"] = pd.to_numeric(_review_signal_frame[source_col], errors="coerce").fillna(0).astype(int)
if "common_signal_total_count" not in _review_signal_frame.columns:
    source_col = "query_safe_signal_total_count" if "query_safe_signal_total_count" in _review_signal_frame.columns else "signal_total_count"
    _review_signal_frame["common_signal_total_count"] = pd.to_numeric(_review_signal_frame[source_col], errors="coerce").fillna(0).astype(int)


def _exact_phrase_hits(text, phrases):
    normalized = normalize_space(text).lower()
    hits = []
    for phrase in phrases:
        if re.search(r"(?<![a-z0-9])" + re.escape(phrase.lower()) + r"(?![a-z0-9])", normalized):
            hits.append(phrase)
    return sorted(set(hits))


def _pipe_union_from_columns(row, columns):
    values = []
    seen = set()
    for col in columns:
        for part in str(row.get(col, "") or "").split(" | "):
            part = normalize_space(part)
            key = part.lower()
            if key and key not in seen:
                seen.add(key)
                values.append(part)
    return " | ".join(values)


def _specific_support_phrases(text):
    normalized = normalize_space(text).lower()
    hits = []
    for need in HERBAL_SPECIFIC_SUPPORT_NEEDS:
        pattern = r"(?<![a-z0-9])" + re.escape(need) + r"\s+support(?![a-z0-9])"
        if re.search(pattern, normalized):
            hits.append(f"{need} support")
    return sorted(set(hits))


def _generic_support_warning(text):
    normalized = normalize_space(text).lower()
    if not re.search(r"\bsupports?\b", normalized):
        return 0
    if _specific_support_phrases(normalized):
        return 0
    return 1


if "sampled_query_cases_df" not in globals() or "query_safe_text" not in sampled_query_cases_df.columns:
    raise RuntimeError(
        "Query-audit annotation requires sampled_query_cases_df.query_safe_text. "
        "Expected sampled_query_cases_df.query_safe_text as the safe source column."
    )

_herbal_review_audit_df = sampled_query_cases_df[["case_id", "query_safe_text"]].copy()
_herbal_review_audit_df["common_category_anchor_terms"] = _herbal_review_audit_df["query_safe_text"].map(
    lambda text: " | ".join(_exact_phrase_hits(text, HERBAL_REVIEW_CATEGORY_ANCHORS))
)
_herbal_review_audit_df["common_generic_utility_terms"] = _herbal_review_audit_df["query_safe_text"].map(
    lambda text: " | ".join(_exact_phrase_hits(text, HERBAL_REVIEW_GENERIC_UTILITY_TOKENS))
)
_herbal_review_audit_df["common_context_dependent_utility_terms"] = _herbal_review_audit_df["query_safe_text"].map(
    lambda text: " | ".join(_exact_phrase_hits(text, HERBAL_REVIEW_CONTEXT_DEPENDENT_TOKENS))
)
_herbal_review_audit_df["common_specific_support_phrases"] = _herbal_review_audit_df["query_safe_text"].map(
    lambda text: " | ".join(_specific_support_phrases(text))
)
_herbal_review_audit_df["common_generic_support_warning"] = _herbal_review_audit_df["query_safe_text"].map(
    _generic_support_warning
).astype(int)
_herbal_review_audit_df["common_form_support_warning"] = _herbal_review_audit_df["query_safe_text"].map(
    lambda text: int(any(
        re.search(
            r"(?<![a-z0-9])" + re.escape(form) + r"\s+support(?![a-z0-9])",
            normalize_space(text).lower(),
        )
        for form in HERBAL_FORM_TOKENS
    ))
)
_herbal_review_audit_df = _herbal_review_audit_df.drop(columns=["query_safe_text"])

_review_signal_frame = _review_signal_frame.merge(
    _herbal_review_audit_df,
    on="case_id",
    how="left",
    validate="one_to_one",
)

_review_signal_frame["common_specific_signal_seed_text"] = _review_signal_frame.apply(
    lambda row: _pipe_union_from_columns(row, SIGNAL_COLUMNS),
    axis=1,
)
_review_signal_frame["query_safe_text"] = (
    _review_signal_frame["query_safe_text"]
    .fillna("")
    .astype(str)
    .map(normalize_space)
)

_fragment_norms = {value.lower() for value in HERBAL_MULTIWORD_ENTITY_FRAGMENT_TOKENS}
_review_signal_frame["common_possible_entity_fragment_signals"] = _review_signal_frame[
    "ingredient_or_herb_signals"
].fillna("").astype(str).map(
    lambda value: " | ".join(
        part
        for part in [normalize_space(v) for v in value.split(" | ")]
        if part and part.lower() in _fragment_norms
    )
)
_review_signal_frame["common_multiword_ingredient_signals"] = _review_signal_frame[
    "ingredient_or_herb_signals"
].fillna("").astype(str).map(
    lambda value: " | ".join(
        part
        for part in [normalize_space(v) for v in value.split(" | ")]
        if part and (" " in part or "'" in part)
    )
)
_review_signal_frame["common_query_audit_policy_version"] = "query_c_audit_v1"

required_common_audit_cols = [
    "common_category_anchor_terms",
    "common_generic_utility_terms",
    "common_context_dependent_utility_terms",
    "common_specific_support_phrases",
    "common_generic_support_warning",
    "common_form_support_warning",
    "common_specific_signal_seed_text",
    "common_possible_entity_fragment_signals",
    "common_multiword_ingredient_signals",
    "common_query_audit_policy_version",
]
missing_common_audit_cols = [col for col in required_common_audit_cols if col not in _review_signal_frame.columns]
if missing_common_audit_cols:
    raise RuntimeError(f"Missing common query-audit columns: {missing_common_audit_cols}")

common_family_rows = []
for family, role in COMMON_REVIEW_SIGNAL_FAMILY_TO_ROLE.items():
    candidate_cols = [
        "qs_" + family + "_signals",
        family + "_signals",
        "has_" + family,
    ]
    present_cols = [col for col in candidate_cols if col in _review_signal_frame.columns]
    for col in present_cols:
        series = _review_signal_frame[col]
        if col.startswith("has_"):
            non_empty = pd.to_numeric(series, errors="coerce").fillna(0).astype(int).gt(0)
            unique_value_count = int(series.nunique(dropna=True))
        else:
            non_empty = series.fillna("").astype(str).str.strip().ne("")
            unique_values = set()
            for value in series.fillna("").astype(str):
                unique_values.update([part.strip().lower() for part in value.split(" | ") if part.strip()])
            unique_value_count = len(unique_values)
        common_family_rows.append({
            "signal_family": family,
            "common_role": role,
            "source_column": col,
            "non_empty_rows": int(non_empty.sum()),
            "coverage_rate": float(non_empty.mean()) if len(non_empty) else 0.0,
            "unique_value_count": int(unique_value_count),
        })

common_review_signal_family_coverage_df = pd.DataFrame(common_family_rows)
globals()[_review_signal_frame_name] = _review_signal_frame

print("Common review signal role coverage:")
if not common_review_signal_family_coverage_df.empty:
    display(common_review_signal_family_coverage_df)
else:
    print("No common signal columns were detected; confirm signal column naming if this is unexpected.")


In [ ]:
# ==== Reconcile Source Cases with the Sampling Manifest ====
from pathlib import Path
import json

if "PROJECT_ROOT" not in globals():
    raise RuntimeError("PROJECT_ROOT is not defined. Run the config cell first.")

sampling_manifest_path = (
    PROJECT_ROOT
    / "data/interim/user_regime_sampling/herbal_user_regime_sampling_manifest.json"
)

print("Rows:", len(sampled_query_cases_df))
print(
    "Regime counts:",
    sampled_query_cases_df["regime"]
    .value_counts()
    .reindex(REGIME_ORDER, fill_value=0)
    .astype(int)
    .to_dict(),
)

print("Sampling manifest exists:", sampling_manifest_path.exists())
print("Sampling manifest path:", sampling_manifest_path)

if sampling_manifest_path.exists():
    with open(sampling_manifest_path, "r", encoding="utf-8") as f:
        sampling_manifest_debug = json.load(f)

    manifest_counts = sampling_manifest_debug.get(
        "eligible_pool_regime_counts_before_query_balance",
        sampling_manifest_debug.get("eligible_pool_regime_counts"),
    )
    manifest_total = sampling_manifest_debug.get(
        "eligible_pool_total_n_before_query_balance",
        sampling_manifest_debug.get("eligible_pool_rows"),
    )

    print("Manifest source counts:", manifest_counts)
    print("Manifest source total:", manifest_total)
    print("Notebook 03 output role:", sampling_manifest_debug.get("notebook_03_output_role"))
    print("Query balance deferred:", sampling_manifest_debug.get("query_balance_deferred_to_query_audit"))
else:
    print("No sampling manifest found; Notebook 05 will validate from parquet only.")


In [ ]:
# ==== Audit Identifier and Entity Cues ====
if 'sampled_query_cases_df' not in globals():
    raise RuntimeError('sampled_query_cases_df is not defined.')

package_identifier_audit_df = pd.DataFrame({
    'checked_rows': [int(len(sampled_query_cases_df))],
    'direct_cue_repaired_rows': [int(sampled_query_cases_df['direct_cue_repair_applied'].sum())],
    'metadata_used_for_positive_signal': [False],
})


In [ ]:
# ==== Audit Direct Cues and Forbidden Evidence ====
if 'sampled_query_cases_df' not in globals():
    raise RuntimeError('sampled_query_cases_df is not defined. Run the input loading cell first.')

if 'review_signals_df' not in globals():
    raise RuntimeError('review_signals_df is not defined. Run the signal extraction cell first.')

direct_leakage_audit_rows = []

brand_audit_cols = [
    'itemctx_facet_brand_text',
]

identifier_audit_cols = [
    'itemctx_identifier_diagnostic_text',
]

required_leakage_audit_cols = [
    'case_id',
    'user_id',
    'target_parent_asin',
    'target_timestamp_ms',
    'query_safe_text',
    *brand_audit_cols,
    *identifier_audit_cols,
    *TITLE_AUDIT_METADATA_COLS,
]

missing_leakage_audit_cols = [
    col for col in required_leakage_audit_cols
    if col not in sampled_query_cases_df.columns
]

if missing_leakage_audit_cols:
    raise RuntimeError(f'Missing leakage audit columns: {missing_leakage_audit_cols}')

for row in sampled_query_cases_df.itertuples(index=False):
    row_dict = row._asdict()
    row_series = pd.Series(row_dict)

    query_safe_text = normalize_space(row_dict['query_safe_text'])

    direct_hits = {
        'contains_asin_pattern': bool(ASIN_PATTERN.search(query_safe_text)),
        'contains_direct_identifier_word': bool(DIRECT_IDENTIFIER_PATTERN.search(query_safe_text)),
        'contains_package_quantity_pattern_warning': bool(PACKAGE_QUANTITY_PATTERN.search(query_safe_text)),
        'contains_seller_manufacturer_pattern': bool(SELLER_MANUFACTURER_PATTERN.search(query_safe_text)),
        'contains_exact_brand_phrase': any_exact_phrase_present(query_safe_text, row_series, brand_audit_cols),
        'contains_exact_identifier_phrase': any_exact_phrase_present(query_safe_text, row_series, identifier_audit_cols),
        'contains_exact_title_phrase_warning': any_exact_phrase_present(
            query_safe_text,
            row_series,
            TITLE_AUDIT_METADATA_COLS,
        ),
        'case_id': row_dict['case_id'],
        'user_id': row_dict['user_id'],
        'target_parent_asin': row_dict['target_parent_asin'],
        'target_timestamp_ms': int(row_dict['target_timestamp_ms']),
    }

    direct_leakage_audit_rows.append(direct_hits)

direct_leakage_audit_df = pd.DataFrame(direct_leakage_audit_rows)

leakage_flag_cols = [
    col for col in direct_leakage_audit_df.columns
    if col.startswith('contains_')
]

blocking_leakage_flag_cols = [
    'contains_asin_pattern',
    'contains_direct_identifier_word',
    'contains_package_quantity_pattern_warning',
    'contains_seller_manufacturer_pattern',
    'contains_exact_brand_phrase',
    'contains_exact_identifier_phrase',
    'contains_exact_title_phrase_warning',
]

warning_leakage_flag_cols = [
    'contains_package_quantity_pattern_warning',
    'contains_exact_title_phrase_warning',
]

missing_blocking_leakage_cols = [
    col for col in blocking_leakage_flag_cols
    if col not in direct_leakage_audit_df.columns
]

missing_warning_leakage_cols = [
    col for col in warning_leakage_flag_cols
    if col not in direct_leakage_audit_df.columns
]

if missing_blocking_leakage_cols:
    raise RuntimeError(f'Missing blocking leakage flag columns: {missing_blocking_leakage_cols}')

if missing_warning_leakage_cols:
    raise RuntimeError(f'Missing warning leakage flag columns: {missing_warning_leakage_cols}')

leakage_flag_count = int(
    direct_leakage_audit_df[blocking_leakage_flag_cols]
    .fillna(False)
    .any(axis=1)
    .sum()
)

package_quantity_warning_count = int(
    direct_leakage_audit_df['contains_package_quantity_pattern_warning']
    .fillna(False)
    .sum()
)

title_exact_leak_warning_count = int(
    direct_leakage_audit_df['contains_exact_title_phrase_warning']
    .fillna(False)
    .sum()
)

review_signals_df = review_signals_df.merge(
    direct_leakage_audit_df[
        ['case_id', 'user_id', 'target_parent_asin', 'target_timestamp_ms'] + leakage_flag_cols
    ],
    on=['case_id', 'user_id', 'target_parent_asin', 'target_timestamp_ms'],
    how='left',
    validate='one_to_one',
)

review_signals_df['direct_cue_warning_count'] = (
    review_signals_df[blocking_leakage_flag_cols]
    .fillna(False)
    .astype(bool)
    .sum(axis=1)
    .astype(int)
)

direct_cue_audit_df = review_signals_df[
    [
        'case_id',
        'user_id',
        'target_parent_asin',
        'regime',
        'direct_cue_repair_applied',
        'direct_cue_repair_count',
        'direct_cue_warning_count',
        *blocking_leakage_flag_cols,
    ]
].copy()

if leakage_flag_count > 0:
    flagged_preview = review_signals_df.loc[
        review_signals_df[blocking_leakage_flag_cols].fillna(False).any(axis=1),
        ['case_id', 'user_id', 'target_parent_asin', 'query_safe_text'] + blocking_leakage_flag_cols,
    ].head(20)

    raise RuntimeError(
        'Direct brand/title/identifier/package shortcut patterns were found in query_safe_text. '
        f'Flagged rows: {leakage_flag_count}. Preview:\n{flagged_preview}'
    )



In [ ]:
# ==== Summarize Signal Coverage and Sufficiency ====
if 'sampled_query_cases_df' not in globals():
    raise RuntimeError('sampled_query_cases_df is not defined. Run the input loading cell first.')

if 'review_signals_df' not in globals():
    raise RuntimeError('review_signals_df is not defined. Run the signal extraction cell first.')

required_summary_objects = [
    'sampled_regime_counts',
    'leakage_flag_count',
    'package_quantity_warning_count',
    'title_exact_leak_warning_count',
    'package_quantity_scrub_summary',
    'query_safe_text_repaired_mask',
    'metadata_matched_rows',
    'metadata_match_rate',
    'itemctx_title_non_empty',
    'itemctx_facet_brand_text_non_empty',
    'itemctx_identifier_diagnostic_text_non_empty',
    'review_reputation_cols_present',
    'facet_policy_values',
    'brand_policy_values',
    'identifier_policy_values',
]

missing_summary_objects = [
    object_name for object_name in required_summary_objects
    if object_name not in globals()
]

if missing_summary_objects:
    raise RuntimeError(f'Missing required summary objects: {missing_summary_objects}')

missing_signal_flag_cols = [
    config['flag_col']
    for config in SIGNAL_FAMILY_CONFIG
    if config['flag_col'] not in review_signals_df.columns
]

if missing_signal_flag_cols:
    raise RuntimeError(f'Missing signal flag columns: {missing_signal_flag_cols}')

signal_family_coverage = {
    config['family']: float(review_signals_df[config['flag_col']].mean())
    for config in SIGNAL_FAMILY_CONFIG
}

query_safe_text_second_pass_repair_count = (
    int(query_safe_text_second_pass_repair_mask.sum())
    if 'query_safe_text_second_pass_repair_mask' in globals()
    else 0
)

review_signal_regime_summary_df = (
    review_signals_df.groupby('regime', dropna=False)
    .agg(
        source_cases=('case_id', 'size'),
        extraction_sufficient_cases=('review_signal_extraction_sufficient', 'sum'),
        mean_signal_family_count=('query_safe_signal_family_count', 'mean'),
        mean_signal_total_count=('query_safe_signal_total_count', 'mean'),
        possible_entity_fragment_cases=('possible_entity_fragment_flag', 'sum'),
        direct_cue_repaired_cases=('direct_cue_repair_applied', 'sum'),
    )
    .reindex(REGIME_ORDER)
    .reset_index()
)
review_signal_regime_summary_df['extraction_insufficient_cases'] = (
    review_signal_regime_summary_df['source_cases']
    - review_signal_regime_summary_df['extraction_sufficient_cases']
)
review_signal_regime_summary_df['sufficient_rate'] = (
    review_signal_regime_summary_df['extraction_sufficient_cases']
    / review_signal_regime_summary_df['source_cases']
)
review_signal_regime_summary_df = review_signal_regime_summary_df[
    [
        'regime',
        'source_cases',
        'extraction_sufficient_cases',
        'extraction_insufficient_cases',
        'sufficient_rate',
        'mean_signal_family_count',
        'mean_signal_total_count',
        'possible_entity_fragment_cases',
        'direct_cue_repaired_cases',
    ]
]

if not review_signal_regime_summary_df['source_cases'].eq(
    review_signal_regime_summary_df['extraction_sufficient_cases']
    + review_signal_regime_summary_df['extraction_insufficient_cases']
).all():
    raise RuntimeError('Extraction summary identity failed by regime.')

extraction_insufficient_reason_summary_df = (
    review_signals_df.groupby(['regime', 'extraction_insufficient_reason'], dropna=False)
    .size()
    .rename('case_count')
    .reset_index()
    .sort_values(['regime', 'case_count'], ascending=[True, False])
)

family_coverage_rows = []
for config in SIGNAL_FAMILY_CONFIG:
    family = config['family']
    family_coverage_rows.append({
        'signal_family': family,
        'signal_column': config['signal_col'],
        'flag_column': config['flag_col'],
        'cases_with_signal': int(review_signals_df[config['flag_col']].astype(bool).sum()),
        'coverage_rate': float(review_signals_df[config['flag_col']].astype(bool).mean()),
        'total_signal_values': int(review_signals_df[f'{family}_signal_count'].sum()),
    })
review_signal_family_coverage_summary_df = pd.DataFrame(family_coverage_rows)

strong_insufficient_case_audit_df = review_signals_df.loc[
    review_signals_df['regime'].eq('strong')
    & ~review_signals_df['review_signal_extraction_sufficient'],
    [
        'case_id',
        'user_id',
        'target_parent_asin',
        'target_rank_desc',
        'regime',
        'target_review_token_count',
        'query_safe_token_count',
        'query_safe_signal_family_count',
        'query_safe_signal_total_count',
        'ingredient_or_herb_signal_count',
        'benefit_need_signal_count',
        'form_signal_count',
        'claim_diet_signal_count',
        'flavor_signal_count',
        'possible_entity_fragment_flag',
        'direct_cue_repair_count',
        'extraction_insufficient_reason',
        *SIGNAL_COLUMNS,
    ],
].copy()
strong_insufficient_case_audit_df['residual_safe_token_count'] = strong_insufficient_case_audit_df['query_safe_token_count']
strong_insufficient_case_audit_df = strong_insufficient_case_audit_df[
    [
        'case_id',
        'user_id',
        'target_parent_asin',
        'target_rank_desc',
        'regime',
        'target_review_token_count',
        'query_safe_token_count',
        'query_safe_signal_family_count',
        'query_safe_signal_total_count',
        'ingredient_or_herb_signal_count',
        'benefit_need_signal_count',
        'form_signal_count',
        'claim_diet_signal_count',
        'flavor_signal_count',
        'residual_safe_token_count',
        'possible_entity_fragment_flag',
        'direct_cue_repair_count',
        'extraction_insufficient_reason',
        *SIGNAL_COLUMNS,
    ]
]

review_signal_summary = {
    'category_label': CATEGORY_LABEL,
    'input_paths': {
        'sampled_query_cases': str(SAMPLED_QUERY_CASES_PATH),
        'item_schema': str(ITEM_SCHEMA_PATH),
    },
    'output_paths': {
        'signals_parquet': str(SIGNAL_PARQUET_PATH),
        'signals_csv': str(SIGNAL_CSV_PATH),
        'summary_json': str(SUMMARY_JSON_PATH),
        'manifest_json': str(MANIFEST_JSON_PATH),
        'regime_summary_csv': str(REGIME_SUMMARY_CSV_PATH),
        'family_coverage_summary_csv': str(FAMILY_COVERAGE_SUMMARY_CSV_PATH),
        'insufficient_reason_summary_csv': str(INSUFFICIENT_REASON_SUMMARY_CSV_PATH),
        'strong_insufficient_audit_csv': str(STRONG_INSUFFICIENT_AUDIT_CSV_PATH),
        'direct_cue_audit_csv': str(DIRECT_CUE_AUDIT_CSV_PATH),
    },
    'input_case_count': int(len(sampled_query_cases_df)),
    'regime_counts': sampled_regime_counts,
    'expected_total_n': int(EXPECTED_TOTAL_N),
    'expected_regime_counts': EXPECTED_REGIME_COUNTS,
    'input_scope': SOURCE_ELIGIBLE_POOL_ROLE,
    'source_eligible_pool_role': SOURCE_ELIGIBLE_POOL_ROLE,
    'source_eligible_pool_total_n': int(EXPECTED_TOTAL_N),
    'source_eligible_pool_regime_counts': EXPECTED_REGIME_COUNTS,
    'source_sampling_contract': (
        'Notebook 03 Query C-lite rank-5 sampling with dynamic balanced source counts from strict training-prior supply, '
        'with target_selection_mode recent_eligible_review_rank_le5.'
    ),
    'target_rank_distribution': {
        str(k): int(v)
        for k, v in sampled_query_cases_df['target_rank_desc']
        .value_counts()
        .sort_index()
        .items()
    },
    'target_selection_mode_distribution': {
        str(k): int(v)
        for k, v in sampled_query_cases_df['target_selection_mode']
        .value_counts()
        .sort_index()
        .items()
    },
    'signal_family_coverage': signal_family_coverage,
    'average_signal_total_count': float(review_signals_df['signal_total_count'].mean()),
    'signal_source': 'heldout_target_review',
    'query_evidence_source': QUERY_EVIDENCE_SOURCE,
    'item_metadata_evidence_used': False,
    'historical_review_evidence_used': False,
    'user_prior_evidence_used': False,
    'target_metadata_fallback_used': False,
    'item_context_fallback_used': False,
    'rating_evidence_used': False,
    'sentiment_evidence_used': False,
    'query_safe_residual_column': 'query_safe_text',
    'cases_with_sufficient_review_signal_extraction': int(
        review_signals_df['review_signal_extraction_sufficient'].sum()
    ),
    'sufficient_review_signal_extraction_by_regime': {
        regime: int(
            review_signals_df.loc[
                review_signals_df['regime'].eq(regime),
                'review_signal_extraction_sufficient',
            ].sum()
        )
        for regime in REGIME_ORDER
    },
    'strong_extraction_insufficient_cases': int(len(strong_insufficient_case_audit_df)),
    'direct_leakage_flagged_rows': int(leakage_flag_count),
    'package_quantity_warning_rows': int(package_quantity_warning_count),
    'package_quantity_policy': PACKAGE_QUANTITY_POLICY,
    'package_quantity_numeric_unit_pattern_version': PACKAGE_QUANTITY_NUMERIC_UNIT_PATTERN_VERSION,
    'package_quantity_rows_removed': int(PACKAGE_QUANTITY_ROWS_REMOVED),
    'package_quantity_text_repair_applied': True,
    'package_quantity_scrub_summary': package_quantity_scrub_summary,
    'exact_title_phrase_warning_rows': int(title_exact_leak_warning_count),
    'query_safe_text_brand_identifier_repair_applied': True,
    'query_safe_text_brand_identifier_repaired_rows': int(query_safe_text_repaired_mask.sum()),
    'query_safe_text_second_pass_repaired_rows': query_safe_text_second_pass_repair_count,
    'direct_cue_removal_policy': (
        'remove_asin_amazon_url_fragments_direct_identifier_words_seller_manufacturer_'
        'and_itemctx_brand_identifier_tokens__package_quantity_numeric_unit_patterns_scrubbed_keep_rows__'
        'remove_exact_item_title_phrase__block_all_remaining_shortcut_flags'
    ),
    'target_review_used_for_signal_extraction': True,
    'query_source': 'target_review_only_query_safe_text_from_notebook_03_revalidated_in_notebook_05',
    'prior_reviews_used_for_query_signal_extraction': False,
    'user_prior_fields_exported': False,
    'item_schema_used_for_direct_cue_removal_only': True,
    'metadata_used_for_query_safety_filtering_only': True,
    'item_metadata_matched_rows': int(metadata_matched_rows),
    'metadata_match_rate': float(metadata_match_rate),
    'itemctx_title_non_empty': int(itemctx_title_non_empty),
    'itemctx_facet_brand_text_non_empty': int(itemctx_facet_brand_text_non_empty),
    'itemctx_identifier_diagnostic_text_non_empty': int(itemctx_identifier_diagnostic_text_non_empty),
    'item_title_used_for_signal_extraction': False,
    'brand_used_for_signal_extraction': False,
    'brand_signal_policy': 'Brand remains metadata-only for direct-cue leakage removal; it is not merged into functional review-derived signals.',
    'identifier_used_for_signal_extraction': False,
    'item_review_reputation_used_for_signal_extraction': False,
    'item_review_reputation_columns_present_but_ignored': review_reputation_cols_present,
    'facet_policy_values': sorted(facet_policy_values),
    'brand_policy_values': sorted(brand_policy_values),
    'identifier_policy_values': sorted(identifier_policy_values),
    'rating_used': False,
    'helpful_vote_used': False,
    'sentiment_used': False,
    'llm_used': False,
    'raw_review_text_exported': False,
    'target_review_text_exported': False,
    'prior_review_text_exported': False,
    'prior_history_loaded': False,
}

print('Input row count:', len(sampled_query_cases_df))
print('Output row count:', len(review_signals_df))
print('Sufficient count:', int(review_signals_df['review_signal_extraction_sufficient'].sum()))
print('Insufficient count:', int((~review_signals_df['review_signal_extraction_sufficient']).sum()))


In [ ]:
# ==== Validate Identity, Evidence, and Leakage Contracts ====
if 'review_signals_df' not in globals():
    raise RuntimeError('review_signals_df is not defined. Run the signal extraction cell first.')

if 'leakage_flag_count' not in globals():
    raise RuntimeError('leakage_flag_count is not defined. Run the leakage audit cell first.')

if 'package_quantity_warning_count' not in globals():
    raise RuntimeError('package_quantity_warning_count is not defined. Run the leakage audit cell first.')

required_validation_signal_cols = [
    'case_id',
    'user_id',
    'target_parent_asin',
    'target_timestamp_ms',
    'target_review_datetime',
    'target_rank_desc',
    'target_selection_mode',
    'regime',
    'query_safe_text',
    'query_safe_residual_text',
    'signal_source',
    'query_evidence_source',
    'item_metadata_evidence_used',
    'historical_review_evidence_used',
    'user_prior_evidence_used',
    'target_metadata_fallback_used',
    'item_context_fallback_used',
    'rating_evidence_used',
    'sentiment_evidence_used',
    'target_review_token_count',
    'query_safe_token_count',
    'query_safe_text_package_quantity_scrub_applied',
    'query_safe_text_package_quantity_scrub_token_delta',
    'signal_family_count',
    'signal_total_count',
    'query_safe_signal_family_count',
    'query_safe_signal_total_count',
    'common_specific_signal_seed_text',
    'common_multiword_ingredient_signals',
    'review_signal_extraction_sufficient',
    'review_only_query_evidence_sufficient',
    'extraction_insufficient_reason',
    'possible_entity_fragment_flag',
    'direct_cue_repair_applied',
    'direct_cue_repair_count',
    'direct_cue_warning_count',
    *SIGNAL_COLUMNS,
    *SIGNAL_FLAG_COLUMNS,
    'contains_asin_pattern',
    'contains_direct_identifier_word',
    'contains_package_quantity_pattern_warning',
    'contains_seller_manufacturer_pattern',
    'contains_exact_brand_phrase',
    'contains_exact_identifier_phrase',
    'contains_exact_title_phrase_warning',
]

missing_validation_signal_cols = [
    col for col in required_validation_signal_cols
    if col not in review_signals_df.columns
]

if missing_validation_signal_cols:
    raise RuntimeError(f'Missing validation signal columns: {missing_validation_signal_cols}')

if len(review_signals_df) != EXPECTED_TOTAL_N:
    raise RuntimeError(f'Expected {EXPECTED_TOTAL_N} signal rows, found {len(review_signals_df)}.')

signal_regime_counts = (
    review_signals_df['regime']
    .value_counts()
    .reindex(REGIME_ORDER, fill_value=0)
    .astype(int)
    .to_dict()
)

if signal_regime_counts != EXPECTED_REGIME_COUNTS:
    raise RuntimeError(
        f'Signal regime counts mismatch: {signal_regime_counts}; expected {EXPECTED_REGIME_COUNTS}.'
    )

if review_signals_df['case_id'].duplicated().any():
    duplicated_case_count = int(review_signals_df['case_id'].duplicated().sum())
    raise RuntimeError(f'Signal output duplicates case_id rows: {duplicated_case_count}')

if set(review_signals_df['case_id'].astype(str)) != set(sampled_query_cases_df['case_id'].astype(str)):
    raise RuntimeError('Signal output case_id set must equal the Notebook 03 source case_id set.')

if review_signals_df['user_id'].duplicated().any():
    duplicated_user_count = int(review_signals_df['user_id'].duplicated().sum())
    raise RuntimeError(f'Signal output duplicates user_id rows: {duplicated_user_count}')

target_key_cols = ['user_id', 'target_parent_asin', 'target_timestamp_ms']

if review_signals_df[target_key_cols].duplicated().any():
    duplicated_target_count = int(review_signals_df[target_key_cols].duplicated().sum())
    raise RuntimeError(
        f'Signal output duplicates selected targets per user: {duplicated_target_count}'
    )

if not review_signals_df['target_rank_desc'].le(MAX_TARGET_RANK_ALLOWED).all():
    raise RuntimeError(f'All target_rank_desc values must be <= {MAX_TARGET_RANK_ALLOWED}.')

if not review_signals_df['target_selection_mode'].astype(str).eq(EXPECTED_TARGET_SELECTION_MODE).all():
    observed_modes = sorted(review_signals_df['target_selection_mode'].dropna().astype(str).unique().tolist())
    raise RuntimeError(
        f'target_selection_mode must be {EXPECTED_TARGET_SELECTION_MODE}; observed {observed_modes}.'
    )

missing_output_target_timestamp_count = int(review_signals_df['target_timestamp_ms'].isna().sum())
if missing_output_target_timestamp_count > 0:
    raise RuntimeError(
        f'target_timestamp_ms must be present and non-null in signal output. Missing rows: {missing_output_target_timestamp_count}.'
    )

if review_signals_df['query_safe_text'].map(normalize_space).eq('').any():
    empty_query_count = int(review_signals_df['query_safe_text'].map(normalize_space).eq('').sum())
    raise RuntimeError(f'query_safe_text must be non-empty in signal output. Empty rows: {empty_query_count}')

if not review_signals_df['signal_source'].eq('heldout_target_review').all():
    raise RuntimeError('signal_source must be heldout_target_review.')

if not review_signals_df['query_evidence_source'].eq(QUERY_EVIDENCE_SOURCE).all():
    raise RuntimeError('query_evidence_source must be target_review_only.')

policy_false_cols = [
    'item_metadata_evidence_used',
    'historical_review_evidence_used',
    'user_prior_evidence_used',
    'target_metadata_fallback_used',
    'item_context_fallback_used',
    'rating_evidence_used',
    'sentiment_evidence_used',
]
for col in policy_false_cols:
    if review_signals_df[col].isna().any():
        raise RuntimeError(f'{col} must not contain missing values.')
    if not pd.api.types.is_bool_dtype(review_signals_df[col]):
        raise RuntimeError(f'{col} must use boolean dtype.')
    if review_signals_df[col].any():
        raise RuntimeError(f'{col} must be False for every output row.')

if review_signals_df['signal_family_count'].lt(0).any():
    raise RuntimeError('signal_family_count must be non-negative.')

if review_signals_df['signal_total_count'].lt(0).any():
    raise RuntimeError('signal_total_count must be non-negative.')

if not review_signals_df['query_safe_signal_family_count'].equals(
    review_signals_df['signal_family_count']
):
    raise RuntimeError(
        'query_safe_signal_family_count must equal signal_family_count.'
    )

if not review_signals_df['query_safe_signal_total_count'].equals(
    review_signals_df['signal_total_count']
):
    raise RuntimeError(
        'query_safe_signal_total_count must equal signal_total_count.'
    )

if PACKAGE_QUANTITY_ROWS_REMOVED != 0:
    raise RuntimeError('Package quantity policy must keep rows; PACKAGE_QUANTITY_ROWS_REMOVED must be 0.')

if leakage_flag_count != 0:
    raise RuntimeError('Final validation failed: direct brand/title/identifier/package shortcut leakage remains after repair.')

blocking_leakage_cols = [
    'contains_asin_pattern',
    'contains_direct_identifier_word',
    'contains_package_quantity_pattern_warning',
    'contains_seller_manufacturer_pattern',
    'contains_exact_brand_phrase',
    'contains_exact_identifier_phrase',
    'contains_exact_title_phrase_warning',
]

if review_signals_df[blocking_leakage_cols].fillna(False).any(axis=1).any():
    raise RuntimeError('Final validation failed: blocking leakage columns contain True values.')

if package_quantity_warning_count != int(
    review_signals_df['contains_package_quantity_pattern_warning'].fillna(False).sum()
):
    raise RuntimeError('package_quantity_warning_count does not match output warning column count.')

if title_exact_leak_warning_count != int(
    review_signals_df['contains_exact_title_phrase_warning'].fillna(False).sum()
):
    raise RuntimeError('title_exact_leak_warning_count does not match output warning column count.')

if package_quantity_warning_count or title_exact_leak_warning_count:
    raise RuntimeError(
        'Package quantity and exact title shortcut flags must be zero after repair.'
    )

for signal_col in SIGNAL_COLUMNS:
    if review_signals_df[signal_col].map(normalize_space).isna().any():
        raise RuntimeError(f'{signal_col} contains invalid null-like values.')

for flag_col in SIGNAL_FLAG_COLUMNS:
    if not review_signals_df[flag_col].isin([0, 1, True, False]).all():
        raise RuntimeError(f'{flag_col} must be binary.')

allowed_policy_audit_cols = set(policy_false_cols)
forbidden_prefixes = [
    'historical_review_',
    'review_reputation_',
    'user_prior_',
    'itemctx_',
]
for col in review_signals_df.columns:
    if col in allowed_policy_audit_cols:
        continue
    if any(str(col).startswith(prefix) for prefix in forbidden_prefixes):
        raise RuntimeError(f'Forbidden evidence-bearing output column present: {col}')

known_complete_multiword_entities = [
    'milk thistle',
    'saw palmetto',
    'olive leaf',
    'dong quai',
    "st john's wort",
    "st. john's wort",
    'ginkgo biloba',
    'holy basil',
    'evening primrose',
    'aloe vera',
    'horny goat weed',
    'red yeast rice',
]
fragment_completion_failures = []
for row in review_signals_df.itertuples(index=False):
    text = normalize_space(getattr(row, 'query_safe_text')).lower()
    signals = normalize_space(getattr(row, 'ingredient_or_herb_signals')).lower()
    signal_values = [value.strip() for value in signals.split('|')]
    for entity in known_complete_multiword_entities:
        entity_norm = normalize_space(entity).lower()
        if entity_norm in text and entity_norm not in signals:
            entity_parts = [part for part in tokenize_text(entity_norm) if part in HERBAL_MULTIWORD_ENTITY_FRAGMENT_TOKENS]
            if any(part in signal_values for part in entity_parts):
                fragment_completion_failures.append(getattr(row, 'case_id'))
                break
if fragment_completion_failures:
    raise RuntimeError(
        'Complete multiword ingredient entity was reduced to a fragment for source-grounded cases: '
        f'{fragment_completion_failures[:20]}'
    )

if not review_signals_df['review_only_query_evidence_sufficient'].equals(
    review_signals_df['review_signal_extraction_sufficient']
):
    raise RuntimeError('Notebook 06 compatibility alias review_only_query_evidence_sufficient must equal review_signal_extraction_sufficient.')

for forbidden_col in [
    'target_review_text',
    'heldout_review_text',
    'prior_review_text',
    'prior_history_n',
    'review_text',
    'review_body',
    'raw_review_text',
    'title',
    'brand',
    'brand_primary',
    'brand_norm',
    'manufacturer',
    'item_model_number',
    'package_dimensions',
    'rating',
    'sentiment',
    'helpful_vote',
    'total_vote',
    'prompt',
    'response',
    'llm_response',
    'query_evidence',
    'canonical_retrieval_text',
    'profile_source_text_dedup_seed',
    'review_reputation_facet_text',
]:
    if forbidden_col in review_signals_df.columns:
        raise RuntimeError(f'Forbidden output column present: {forbidden_col}')

print('Validation status: PASS')


In [ ]:
# ==== Preview Residual Warning Cases ====
if 'review_signals_df' in globals() and 'contains_package_quantity_pattern_warning' in review_signals_df.columns:
    display(
        review_signals_df.loc[
            review_signals_df["contains_package_quantity_pattern_warning"].fillna(False),
            [
                "case_id",
                "user_id",
                "target_parent_asin",
                "regime",
                "query_safe_text",
                "contains_package_quantity_pattern_warning",
                "query_safe_text_package_quantity_scrub_applied",
                "query_safe_text_package_quantity_scrub_token_delta",
            ],
        ].head(20)
    )


In [ ]:
# ==== Export Signal Tables and Audit Summaries ====
if 'review_signals_df' not in globals():
    raise RuntimeError('review_signals_df is not defined. Run the signal extraction and validation cells first.')

if 'review_signal_summary' not in globals():
    raise RuntimeError('review_signal_summary is not defined. Run the summary cell first.')

required_save_paths = [
    SIGNAL_PARQUET_PATH,
    SIGNAL_CSV_PATH,
    SUMMARY_JSON_PATH,
    MANIFEST_JSON_PATH,
    REGIME_SUMMARY_CSV_PATH,
    FAMILY_COVERAGE_SUMMARY_CSV_PATH,
    INSUFFICIENT_REASON_SUMMARY_CSV_PATH,
    STRONG_INSUFFICIENT_AUDIT_CSV_PATH,
    DIRECT_CUE_AUDIT_CSV_PATH,
]

for output_path in required_save_paths:
    output_path.parent.mkdir(parents=True, exist_ok=True)

if len(review_signals_df) != EXPECTED_TOTAL_N:
    raise RuntimeError(
        f'Refusing to save: expected {EXPECTED_TOTAL_N} signal rows, found {len(review_signals_df)}.'
    )

if review_signals_df['case_id'].duplicated().any():
    duplicated_case_count = int(review_signals_df['case_id'].duplicated().sum())
    raise RuntimeError(f'Refusing to save: duplicated case_id rows: {duplicated_case_count}')

if review_signal_summary.get('input_case_count') != int(len(review_signals_df)):
    raise RuntimeError(
        'Refusing to save: summary input_case_count does not match review_signals_df row count.'
    )

for stale_output_path in required_save_paths + [
    SIGNAL_PARQUET_PATH,
    SIGNAL_CSV_PATH,
    SIGNAL_PARQUET_COMPAT_PATH,
    SIGNAL_CSV_COMPAT_PATH,
]:
    if stale_output_path.exists():
        stale_output_path.unlink()

review_signals_df.to_parquet(SIGNAL_PARQUET_PATH, index=False)
review_signals_df.to_csv(SIGNAL_CSV_PATH, index=False, encoding='utf-8-sig')
review_signals_df.to_parquet(SIGNAL_PARQUET_COMPAT_PATH, index=False)
review_signals_df.to_csv(SIGNAL_CSV_COMPAT_PATH, index=False, encoding='utf-8-sig')

review_signal_regime_summary_df.to_csv(REGIME_SUMMARY_CSV_PATH, index=False, encoding='utf-8-sig')
review_signal_family_coverage_summary_df.to_csv(FAMILY_COVERAGE_SUMMARY_CSV_PATH, index=False, encoding='utf-8-sig')
extraction_insufficient_reason_summary_df.to_csv(INSUFFICIENT_REASON_SUMMARY_CSV_PATH, index=False, encoding='utf-8-sig')
strong_insufficient_case_audit_df.to_csv(STRONG_INSUFFICIENT_AUDIT_CSV_PATH, index=False, encoding='utf-8-sig')
direct_cue_audit_df.to_csv(DIRECT_CUE_AUDIT_CSV_PATH, index=False, encoding='utf-8-sig')

with open(SUMMARY_JSON_PATH, 'w', encoding='utf-8') as f:
    json.dump(review_signal_summary, f, ensure_ascii=False, indent=2, default=str)

final_regime_counts = (
    review_signals_df['regime']
    .value_counts()
    .reindex(REGIME_ORDER, fill_value=0)
    .astype(int)
    .to_dict()
)
final_target_selection_modes = sorted(
    review_signals_df['target_selection_mode'].dropna().astype(str).unique().tolist()
)
final_missing_target_timestamp_count = int(review_signals_df['target_timestamp_ms'].isna().sum())

review_signal_manifest = {
    'input_scope': SOURCE_ELIGIBLE_POOL_ROLE,
    'signal_source': 'heldout_target_review',
    'query_evidence_source': QUERY_EVIDENCE_SOURCE,
    'item_metadata_evidence_used': False,
    'historical_review_evidence_used': False,
    'user_prior_evidence_used': False,
    'target_metadata_fallback_used': False,
    'item_context_fallback_used': False,
    'raw_review_text_exported': False,
    'input_case_count': int(len(sampled_query_cases_df)),
    'output_case_count': int(len(review_signals_df)),
    'regime_counts': final_regime_counts,
    'extraction_sufficient_by_regime': {
        regime: int(review_signals_df.loc[review_signals_df['regime'].eq(regime), 'review_signal_extraction_sufficient'].sum())
        for regime in REGIME_ORDER
    },
    'strong_extraction_insufficient_cases': int(len(strong_insufficient_case_audit_df)),
    'output_paths': review_signal_summary['output_paths'],
}
with open(MANIFEST_JSON_PATH, 'w', encoding='utf-8') as f:
    json.dump(review_signal_manifest, f, ensure_ascii=False, indent=2, default=str)

print('Input path:', SAMPLED_QUERY_CASES_PATH)
print('Output path:', SIGNAL_PARQUET_COMPAT_PATH)
print('Input row count:', len(sampled_query_cases_df))
print('Output row count:', len(review_signals_df))
print('Sufficient count:', int(review_signals_df['review_signal_extraction_sufficient'].sum()))
print('Insufficient count:', int((~review_signals_df['review_signal_extraction_sufficient']).sum()))
print('Validation status: PASS')


In [ ]:
# ==== Export the Cross-Category Review-Signal Contract ====
common_review_signal_contract = dict(COMMON_REVIEW_SIGNAL_CONTRACT)
_review_signal_frame = globals().get("review_signals_df")
common_review_signal_contract["output_paths"] = {
    "common_review_signal_contract": str(OUTPUT_DIR / "herbal_review_signal_common_contract.json"),
    "common_review_signal_family_coverage": str(OUTPUT_DIR / "herbal_review_signal_common_family_coverage.csv"),
}
common_review_signal_contract["row_count"] = int(len(_review_signal_frame)) if _review_signal_frame is not None else None
common_review_signal_contract["columns"] = list(_review_signal_frame.columns) if _review_signal_frame is not None else []
common_review_signal_contract["query_evidence_contract"] = {
    "signal_source": "heldout_target_review",
    "query_evidence_source": QUERY_EVIDENCE_SOURCE,
    "item_metadata_evidence_used": False,
    "historical_review_evidence_used": False,
    "user_prior_evidence_used": False,
    "target_metadata_fallback_used": False,
    "item_context_fallback_used": False,
    "rating_evidence_used": False,
    "sentiment_evidence_used": False,
    "query_safe_residual_column": "query_safe_text",
    "specific_signal_seed_column": "common_specific_signal_seed_text",
}
common_review_signal_contract["query_audit_fields"] = [
    "common_category_anchor_terms",
    "common_generic_utility_terms",
    "common_context_dependent_utility_terms",
    "common_specific_support_phrases",
    "common_generic_support_warning",
    "common_form_support_warning",
    "common_specific_signal_seed_text",
    "common_possible_entity_fragment_signals",
    "common_multiword_ingredient_signals",
]
common_review_signal_contract["downstream_note"] = (
    "Existing signal columns are preserved. Notebook 06 may optionally use "
    "common_specific_signal_seed_text for a cleaned Query C sensitivity run. "
    "If multiword entities fragment only after this stage, inspect Notebook 06."
)

with open(OUTPUT_DIR / "herbal_review_signal_common_contract.json", "w", encoding="utf-8") as f:
    json.dump(common_review_signal_contract, f, ensure_ascii=False, indent=2)

if "common_review_signal_family_coverage_df" in globals() and not common_review_signal_family_coverage_df.empty:
    common_review_signal_family_coverage_df.to_csv(OUTPUT_DIR / "herbal_review_signal_common_family_coverage.csv", index=False, encoding="utf-8-sig")

print("Saved common review signal contract:", OUTPUT_DIR / "herbal_review_signal_common_contract.json")
if "common_review_signal_family_coverage_df" in globals() and not common_review_signal_family_coverage_df.empty:
    print("Saved common review signal family coverage:", OUTPUT_DIR / "herbal_review_signal_common_family_coverage.csv")
